# Assignment 2 (PeopleNet), Phase 1: backbone comparison

**Goal:** find the pre-trained backbone whose features best separate *person* from *non-person* images,
measured by **validation accuracy at the 0.5 threshold** (the graded metric).

**Method (cheap on purpose):**
1. Decode the 100K images from the zip **once** into RAM (optionally cached in `/tmp`).
2. For each backbone (frozen): run every image through it once and store the pooled features.
3. Train small heads (linear, small MLP) on the stored features. Each head trains in seconds.
4. Rank backbones by validation accuracy. The best one goes to Phase 2 (fine-tuning).
5. As a safety net, assemble the best backbone and head into one Keras model and write all
   deliverables to `pnp_model/`, so a valid submission exists even before Phase 2.

**Two fixes over the template:**
* Each backbone gets its own preprocessing (`keras_hub` `ImageConverter`: resize and ImageNet normalization).
  The template fed raw 0..255 pixels into the backbone.
* The output layer has a sigmoid. The template's `Dense(1)` produced logits, which made
  `binary_crossentropy` meaningless (loss 8.0, accuracy 0.50) and produced outputs outside [0, 1].

**Instance:** `ml.g4dn.4xlarge` (64 GB RAM, T4 GPU). The decoded train set alone is about 12 GB.
To check that the code runs on a cheap CPU instance first, set `LIMIT = 2000` and keep only `efficientnet_b0_ra_imagenet` in `BACKBONES`.

In [ ]:
import os
# Keras backend: must be set before keras is imported.
os.environ.setdefault("KERAS_BACKEND", "tensorflow")

# Backbones to compare (keras_hub preset names). Each is run once over all 100K images.
# Comment out the ones you don't want to pay for. Rough cost order: top = cheapest.
BACKBONES = [
    'efficientnet_b0_ra_imagenet',        # 224px, 4M params   (template family: the baseline)
    'efficientnet_b3_ra2_imagenet',       # 288px, 11M params
    'efficientnet2_rw_s_ra2_imagenet',    # 288px, 22M params  (EfficientNetV2-S)
    'hgnetv2_b5_ssld_stage1_in22k_in1k',  # 224px, 33M params  (ImageNet-22k pre-training)
    'dinov3_vit_base_lvd1689m',           # 224px, 86M params  (self-supervised ViT)
    'clip_vit_base_patch16',              # 224px, 86M params  (image-text ViT, vision tower only)
    'siglip2_base_patch16_224',           # 224px, 86M params  (image-text ViT, vision tower only)
    'efficientnet_b5_sw_ft_imagenet',     # 448px, 29M params  (slowest: 4x the pixels)
]
# Deliberately NOT used: presets trained on COCO with labels (dfine_*_coco, retinanet_*_coco).
# This dataset is built from COCO, so those detectors may have seen these very images with "person" boxes.

HEADS = ['linear', 'mlp']

LIMIT = None              # e.g. 2000 to prototype on a CPU instance; None = the full dataset
VAL_FRACTION = 0.1        # 8K labeled images held out for model selection
SEED = 11
EXTRACT_BATCH = 64
MIXED_PRECISION = True    # float16 on the T4 tensor cores: about 2-3x faster extraction

WORK_DIR = 'pnp_phase1'          # persistent home volume (10 GB): small files only
CACHE_DIR = '/tmp/pnp_cache'     # large files (pixels, features); lost when the space stops
os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(CACHE_DIR, exist_ok=True)

In [ ]:
!ls pnp_dataset.zip || (aws s3 cp s3://danylo-ucla/pnp_dataset.zip ./)

In [ ]:
# GPU configuration (from the template).
import keras
import tensorflow as tf
print("Keras backend:", keras.backend.backend(), "| Keras", keras.__version__)
for gpu in tf.config.list_physical_devices('GPU'):
    print(gpu)
    tf.config.experimental.set_memory_growth(gpu, True)

import json, time, shutil, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import PIL.Image
import keras_hub
from concurrent.futures import ThreadPoolExecutor
import warnings
# Harmless: Keras warns when a finite tf.data pipeline ends, and about input-name structure in some keras_hub models.
warnings.filterwarnings('ignore', message='.*ran out of data.*')
warnings.filterwarnings('ignore', message='.*structure of `inputs`.*')
print("keras_hub", keras_hub.__version__)

## Data
Images are decoded once with parallel threads and kept as `uint8` (4x smaller than float32).
The order matches the template's `build_dataset` exactly (sorted file names), so labels and
score rows line up.

In [ ]:
def list_images(z: zipfile.ZipFile, segment: str) -> list:
    # Same selection and order as collegium.m02_cnn.utils.pnp_dataset.load_images
    return sorted(p for p in z.namelist()
                  if p.startswith(f'pnp_dataset/{segment}') and 'npy' not in p and not p.endswith('/'))


def _decode_chunk(names, out, start):
    with zipfile.ZipFile('pnp_dataset.zip') as z:
        for k, name in enumerate(names):
            with z.open(name) as f:
                img = np.asarray(PIL.Image.open(f).convert('RGB'), dtype=np.uint8)
            assert img.shape == (224, 224, 3), (name, img.shape)
            out[start + k] = img


def decode_segment(segment: str, limit=None) -> np.ndarray:
    cache = f'{CACHE_DIR}/{segment}_x_{limit or "all"}.npy'
    if os.path.exists(cache):
        print(f'{segment}: loading cached pixels from {cache}')
        return np.load(cache, mmap_mode='r')

    with zipfile.ZipFile('pnp_dataset.zip') as z:
        names = list_images(z, segment)
    if limit is not None:
        names = names[:limit]

    out = np.empty((len(names), 224, 224, 3), dtype=np.uint8)
    chunk = 500
    t0 = time.time()
    with ThreadPoolExecutor(max_workers=os.cpu_count()) as ex:
        list(ex.map(lambda i: _decode_chunk(names[i:i + chunk], out, i), range(0, len(names), chunk)))
    print(f'{segment}: decoded {len(names)} images in {time.time() - t0:.0f}s ({out.nbytes / 1e9:.1f} GB)')

    # Cache to /tmp if there is room, so a kernel restart does not need to decode again.
    if shutil.disk_usage(CACHE_DIR).free > out.nbytes * 1.2:
        np.save(cache, out)
    else:
        print(f'  not cached: not enough free space in {CACHE_DIR}')
    return out


train_x = decode_segment('train', LIMIT)
score_x = decode_segment('score', None if LIMIT is None else min(LIMIT, 1000))

with zipfile.ZipFile('pnp_dataset.zip') as z:
    with z.open('pnp_dataset/train_y.npy') as f:
        train_y = np.load(f).astype(np.int32)
train_y = train_y[:len(train_x)]

print('train', train_x.shape, 'score', score_x.shape)
print('person share in train:', train_y.mean().round(4))

In [ ]:
from sklearn.model_selection import train_test_split

# Stratified split: the validation set has the same person / non-person mix as the training set.
# The split is saved so that Phase 2 uses exactly the same validation images.
split_path = f'{WORK_DIR}/split.npz'
if os.path.exists(split_path) and len(np.load(split_path)['train_idx']) + len(np.load(split_path)['val_idx']) == len(train_x):
    s = np.load(split_path); tr_idx, va_idx = s['train_idx'], s['val_idx']
    print('reusing saved split')
else:
    tr_idx, va_idx = train_test_split(np.arange(len(train_x)), test_size=VAL_FRACTION,
                                      stratify=train_y, random_state=SEED)
    tr_idx, va_idx = np.sort(tr_idx), np.sort(va_idx)
    np.savez(split_path, train_idx=tr_idx, val_idx=va_idx)

print(f'fit: {len(tr_idx)} images ({train_y[tr_idx].mean():.3f} person) | '
      f'val: {len(va_idx)} images ({train_y[va_idx].mean():.3f} person)')

In [ ]:
# A look at the data: 8 person and 8 non-person images.
fig, axes = plt.subplots(2, 8, figsize=(16, 4.5))
for row, label in enumerate([1, 0]):
    idx = np.flatnonzero(train_y == label)[:8]
    for ax, i in zip(axes[row], idx):
        ax.imshow(train_x[i]); ax.axis('off')
    axes[row][0].set_title('person' if label else 'non-person', loc='left')
plt.tight_layout(); plt.show()

## Feature extraction
For every backbone, one model maps raw pixels (224x224x3, 0..255) to a feature vector:

`pixels → ImageConverter (resize + normalize) → frozen backbone → pooling`

* **CNNs** (EfficientNet, HGNetV2): global **average** and **max** pooling, concatenated.
  Max pooling keeps a strong local response, which helps with small people in large scenes.
* **ViTs** (DINOv3, CLIP): the CLS token plus the mean of all tokens.
* **SigLIP**: its own pooled image embedding.

Only built-in Keras layers are used around the backbone, so the final model loads without custom code.

In [ ]:
def build_extractor(preset: str, load_weights: bool = True) -> keras.Model:
    # On a GPU, Keras compiles predict() with XLA, which has no bicubic or antialiased resize.
    # Images are already 224x224: skip resizing when the backbone expects 224, otherwise
    # upsample with plain bilinear. Normalization (scale/offset) is unchanged.
    native = tuple(keras_hub.layers.ImageConverter.from_preset(preset).get_config()['image_size'])
    if native == (224, 224):
        converter = keras_hub.layers.ImageConverter.from_preset(preset, image_size=None)
    else:
        converter = keras_hub.layers.ImageConverter.from_preset(preset, interpolation='bilinear', antialias=False)
    encoder =keras_hub.models.Backbone.from_preset(preset, load_weights=load_weights)
    if hasattr(encoder, 'vision_encoder'):        # CLIP / SigLIP: keep the image tower only
        encoder = encoder.vision_encoder

    pixels = keras.Input((224, 224, 3), name='pixels')
    y = encoder(converter(pixels))
    if isinstance(y, dict):                       # e.g. HGNetV2 returns every stage: keep the last
        y = y[list(y)[-1]]

    if len(y.shape) == 4:                         # CNN feature map (B, H, W, C)
        feats = keras.layers.Concatenate(name='features')([
            keras.layers.GlobalAveragePooling2D()(y), keras.layers.GlobalMaxPooling2D()(y)])
    elif len(y.shape) == 3:                       # ViT tokens (B, T, C): CLS token + mean token
        cls = keras.layers.Flatten()(keras.layers.Cropping1D((0, y.shape[1] - 1))(y))
        feats = keras.layers.Concatenate(name='features')([cls, keras.layers.GlobalAveragePooling1D()(y)])
    else:                                         # already pooled (B, C)
        feats = y

    extractor = keras.Model(pixels, feats, name='extractor')
    extractor.trainable = False
    return extractor


def describe(preset: str, extractor: keras.Model) -> dict:
    conv = keras_hub.layers.ImageConverter.from_preset(preset).get_config()
    return {
        'image_size': conv.get('image_size'),
        'params_M': round(extractor.count_params() / 1e6, 1),
        'feature_dim': int(extractor.output.shape[-1]),
    }


def batches(x: np.ndarray, idx=None, batch_size=EXTRACT_BATCH):
    idx = np.arange(len(x)) if idx is None else idx
    def gen():
        for i in range(0, len(idx), batch_size):
            yield x[idx[i:i + batch_size]].astype(np.float32)
    return tf.data.Dataset.from_generator(
        gen, output_signature=tf.TensorSpec((None, 224, 224, 3), tf.float32)).prefetch(4)

In [ ]:
def extract(preset: str) -> dict:
    """Runs one backbone over train and score images; stores float16 features in CACHE_DIR."""
    tag = f'{preset}_{LIMIT or "all"}'
    meta_path = f'{WORK_DIR}/{tag}_meta.json'
    paths = {s: f'{CACHE_DIR}/{tag}_{s}.npy' for s in ['train', 'score']}
    if os.path.exists(meta_path) and all(os.path.exists(p) for p in paths.values()):
        print(f'{preset}: features already extracted')
        return json.load(open(meta_path))

    # Try float16 first (faster); some models overflow in float16 (DINOv3 returns NaN),
    # so a probe batch decides, and the backbone falls back to float32 if needed.
    for precision in (['mixed_float16', 'float32'] if MIXED_PRECISION else ['float32']):
        keras.config.set_dtype_policy(precision)
        try:
            extractor = build_extractor(preset)
        finally:
            keras.config.set_dtype_policy('float32')
        probe = extractor.predict(train_x[:EXTRACT_BATCH].astype(np.float32), verbose=0)
        if np.isfinite(np.asarray(probe, dtype=np.float32)).all():
            break
        print(f'{preset}: non-finite features in {precision}, retrying in float32')
        del extractor
        keras.backend.clear_session()
    meta = {'preset': preset, 'precision': precision, **describe(preset, extractor)}
    print(f'== {preset}: {meta}')

    t0 = time.time()
    for segment, x in [('train', train_x), ('score', score_x)]:
        f = extractor.predict(batches(x), verbose=1).astype(np.float16)
        assert len(f) == len(x) and np.isfinite(f).all(), f'{preset}: bad features on {segment}'
        np.save(paths[segment], f)
    meta['extract_sec'] = round(time.time() - t0)
    meta['images_per_sec'] = round((len(train_x) + len(score_x)) / meta['extract_sec'], 1)
    json.dump(meta, open(meta_path, 'w'))

    del extractor
    keras.backend.clear_session()
    return meta


metas = {}
for preset in BACKBONES:
    metas[preset] = extract(preset)
pd.DataFrame(metas.values())

## Heads on stored features
* **linear**: standardize features → dropout → `Dense(1, sigmoid)` (a linear probe)
* **mlp**: standardize → dropout → `Dense(512, gelu)` → dropout → `Dense(1, sigmoid)`

Early stopping on validation accuracy. We also report the best threshold on validation:
if it is far from 0.5, the final layer's bias can be shifted so that the model's own 0.5 cutoff
lands there.

In [ ]:
def build_head(kind: str, train_feats: np.ndarray) -> keras.Model:
    norm = keras.layers.Normalization(name='feature_norm')
    norm.adapt(train_feats)
    layers = [keras.Input(train_feats.shape[1:]), norm]
    if kind == 'linear':
        layers += [keras.layers.Dropout(0.2)]
    elif kind == 'mlp':
        layers += [keras.layers.Dropout(0.3), keras.layers.Dense(512, activation='gelu'), keras.layers.Dropout(0.3)]
    else:
        raise ValueError(kind)
    layers += [keras.layers.Dense(1, activation='sigmoid', dtype='float32', name='person_prob')]
    head = keras.Sequential(layers, name=f'head_{kind}')
    head.compile(optimizer=keras.optimizers.Adam(1e-3), loss='binary_crossentropy',
                 metrics=[keras.metrics.BinaryAccuracy(name='acc', threshold=0.5)])
    return head


def best_threshold(y: np.ndarray, p: np.ndarray):
    ts = np.round(np.linspace(0.05, 0.95, 181), 3)
    accs = np.array([((p >= t) == y).mean() for t in ts])
    best = accs.max()
    t = ts[accs == best][np.argmin(np.abs(ts[accs == best] - 0.5))]   # ties: closest to 0.5
    return float(t), float(best)


def train_head(kind: str, f_tr, y_tr, f_va, y_va):
    keras.utils.set_random_seed(SEED)
    head = build_head(kind, f_tr)
    hist = head.fit(f_tr, y_tr, validation_data=(f_va, y_va), epochs=60, batch_size=512, verbose=0,
                    callbacks=[keras.callbacks.EarlyStopping(monitor='val_acc', mode='max', patience=6,
                                                             restore_best_weights=True),
                               keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3)])
    p_va = head.predict(f_va, batch_size=2048, verbose=0).ravel()
    return head, p_va, len(hist.history['loss'])

In [ ]:
results = []
for preset in BACKBONES:
    tag = f'{preset}_{LIMIT or "all"}'
    feats = np.load(f'{CACHE_DIR}/{tag}_train.npy').astype(np.float32)
    f_tr, f_va, y_tr, y_va = feats[tr_idx], feats[va_idx], train_y[tr_idx], train_y[va_idx]
    for kind in HEADS:
        head, p_va, epochs = train_head(kind, f_tr, y_tr, f_va, y_va)
        t, acc_t = best_threshold(y_va, p_va)
        head.save(f'{CACHE_DIR}/{tag}_head_{kind}.keras')
        row = {'backbone': preset, 'head': kind,
               'val_acc@0.5': round(float(((p_va >= 0.5) == y_va).mean()), 4),
               'best_t': t, 'val_acc@best_t': round(acc_t, 4), 'epochs': epochs,
               **{k: metas[preset].get(k) for k in ['image_size', 'params_M', 'feature_dim', 'extract_sec']}}
        results.append(row)
        print(row)
    del feats, f_tr, f_va
    keras.backend.clear_session()

results = pd.DataFrame(results).sort_values('val_acc@0.5', ascending=False).reset_index(drop=True)
results.to_csv(f'{WORK_DIR}/phase1_results.csv', index=False)
results

In [ ]:
# Best head per backbone, as a chart.
best_per_bb = results.groupby('backbone')['val_acc@0.5'].max().sort_values()
ax = best_per_bb.plot.barh(figsize=(8, 0.45 * len(best_per_bb) + 1), color='#4c72b0')
ax.set_xlim(max(0.5, best_per_bb.min() - 0.03), 1.0)
ax.set_xlabel('validation accuracy @ 0.5 (frozen backbone, best head)')
for i, v in enumerate(best_per_bb.values):
    ax.text(v + 0.002, i, f'{v:.3f}', va='center')
plt.tight_layout(); plt.show()

## Safety-net submission (best frozen backbone + head)
This writes a complete, valid submission to `pnp_model/`, so you have something to hand in before Phase 2.
The final model is a single Keras model: raw pixels in, person probability out.
If the best validation threshold is not 0.5, it is folded into the last layer's bias:
`sigmoid(z - logit(t))` crosses 0.5 exactly where `sigmoid(z)` crosses `t`.

In [ ]:
FOLD_THRESHOLD = True   # set False to keep the plain 0.5 cutoff

best = results.iloc[0]
preset, kind = best['backbone'], best['head']
print('Best:', dict(best))
tag = f'{preset}_{LIMIT or "all"}'

head = keras.models.load_model(f'{CACHE_DIR}/{tag}_head_{kind}.keras')
t = best['best_t'] if FOLD_THRESHOLD and best['val_acc@best_t'] > best['val_acc@0.5'] else 0.5
if t != 0.5:
    out = head.get_layer('person_prob')
    w, b = out.get_weights()
    out.set_weights([w, b - np.log(t / (1 - t))])
    print(f'folded threshold {t} into the output bias')

# float32 end to end, so the model behaves the same on any machine (GPU or CPU).
extractor = build_extractor(preset)
model = keras.Model(extractor.input, head(extractor.output), name='pnp_model')
model.summary(line_length=110)

# Sanity check: the assembled model reproduces the head's validation accuracy.
check = va_idx[:2000]
p_check = model.predict(batches(train_x, check), verbose=0).ravel()
print(f'assembled model, val accuracy @0.5 on {len(check)} images:', ((p_check >= 0.5) == train_y[check]).mean().round(4))

In [ ]:
model_dir = 'pnp_model'
os.makedirs(model_dir, exist_ok=True)

# Probabilities of the person label for the score segment, in the original order.
score_p = model.predict(batches(score_x), verbose=1).astype(np.float32)
score_y_hat = pd.DataFrame(score_p, columns=['score'])
assert score_y_hat['score'].dtype == np.float32 and score_y_hat['score'].between(0, 1).all()
if LIMIT is None:
    assert score_y_hat.shape == (20000, 1)
score_y_hat.to_parquet(f'{model_dir}/score_y_hat.parquet')

with open(f'{model_dir}/model_architecture.json', 'w') as f:
    f.write(json.dumps(json.loads(model.to_json()), indent=True))
keras.models.save_model(model, f'{model_dir}/model.keras')

# The saved file must reproduce the submitted scores.
reloaded = keras.models.load_model(f'{model_dir}/model.keras')
p_re = reloaded.predict(score_x[:256].astype(np.float32), verbose=0).ravel()
print('max abs diff vs submitted scores:', np.abs(p_re - score_y_hat['score'].values[:256]).max())
assert np.allclose(p_re, score_y_hat['score'].values[:256], atol=1e-3)
print('predicted person share on score:', (score_y_hat['score'] >= 0.5).mean().round(4))
print(os.listdir(model_dir))

## What to send back for Phase 2
1. The `results` table (or `pnp_phase1/phase1_results.csv`).
2. The `extract_sec` per backbone (it tells us what fine-tuning will cost).
3. Anything that failed or looked odd.

Phase 2 fine-tunes the top one or two backbones end to end (with augmentation), on the same
validation split (`pnp_phase1/split.npz`).

**Remember to stop the space when you are done** (the GPU instance costs 7.5x the base rate).